# Scientific Calculator Agent

This project is a minimal introduction to Agentic AI using scientific
and cheminformatics tools.

We will progressively transform ordinary Python functions into a
small LLM-powered agent.

## Learning progression

1. Build ordinary Python scientific tools.
2. Make a basic LLM call.
3. Give the LLM access to our tools through tool calling.
4. Build a mini agent that can decide which tool to use, execute it,
   return the result to the LLM, and produce a final response.

## Scientific tools

We will implement:

- `calculate_area(length, width)`
- `calculate_molecular_weight(smiles)`

The first performs a deterministic mathematical calculation.

The second uses RDKit to calculate molecular weight from a SMILES string.

## Important principle

The LLM does not perform the scientific calculation itself.

Instead:

    User
      ↓
    LLM → selects the appropriate tool
      ↓
    Python / RDKit → performs the calculation
      ↓
    LLM → interprets the result
      ↓
    Final answer

This separation between reasoning/orchestration and deterministic
scientific computation is a central design principle of the projects
that follow.

# 1. Environment Setup

This notebook runs inside the project-specific `.venv` environment.

We will verify:

- Python version
- Python executable
- RDKit installation
- Groq SDK
- python-dotenv
- ipykernel

The Groq API will be introduced in Exercise 2.

In [3]:
import sys
import rdkit
import groq
import dotenv
import ipykernel

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

print("\nRDKit version:")
print(rdkit.__version__)

print("\nGroq SDK:")
print(groq.__version__)

print("\npython-dotenv:")
print("OK")

print("\nipykernel:")
print("OK")

Python version:
3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]

Python executable:
c:\Users\HP\Documents\Agentic_AI\Agentic-AI-for-Scientific-Discovery\00_Scientific_Calculator_Agent\.venv\Scripts\python.exe

RDKit version:
2026.03.6

Groq SDK:
1.7.0

python-dotenv:
OK

ipykernel:
OK


## 1.1 Load the environment variables

The Groq API key is stored in `.env`.

We will load it into the Python environment without displaying
the key itself.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if api_key:
    print("GROQ_API_KEY loaded successfully.")
    print("Key length:", len(api_key))
else:
    raise ValueError(
        "GROQ_API_KEY was not found. "
        "Check your .env file."
    )

GROQ_API_KEY loaded successfully.
Key length: 56


# 2. Exercise 1 — Python Scientific Tools

Before introducing an LLM, we first build the capabilities that the
future agent will use.

This is deliberately ordinary Python.

At this stage there is:

- no AI
- no LLM
- no tool calling
- no agent

The important idea is:

    Scientific capability first
    Agentic orchestration later

## 2.1 Calculate Area

Our first scientific tool is a simple deterministic function.

It takes:

- length
- width

and returns:

- area

In [5]:
def calculate_area(length, width):
    """Calculate the area of a rectangle."""
    
    return length * width

In [6]:
area = calculate_area(10, 5)

print("Area:", area)

Area: 50


## 2.2 Calculate Molecular Weight

Our second tool is a cheminformatics function.

The user provides a molecule as a SMILES string.

RDKit will:

1. Parse the SMILES.
2. Create a molecular representation.
3. Calculate the molecular weight.

The calculation is performed by RDKit, not by the LLM.

In [7]:
from rdkit import Chem
from rdkit.Chem import Descriptors

def calculate_molecular_weight(smiles):
    """Calculate molecular weight from a SMILES string."""
    
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    
    return Descriptors.MolWt(mol)

In [8]:
smiles = "CCO"  # Ethanol

mw = calculate_molecular_weight(smiles)

print("SMILES:", smiles)
print("Molecular weight:", mw)

SMILES: CCO
Molecular weight: 46.069


## 2.3 Test the Scientific Tools

Before turning these functions into tools for an LLM, we verify that
they work correctly on their own.

In [9]:
# Test area
assert calculate_area(10, 5) == 50

# Test molecular weight
ethanol_mw = calculate_molecular_weight("CCO")
assert round(ethanol_mw, 2) == 46.07

print("All basic tests passed.")

All basic tests passed.


In [10]:
try:
    calculate_molecular_weight("not_a_smiles")
except ValueError as e:
    print("Error:", e)

Error: Invalid SMILES: not_a_smiles


[11:49:35] SMILES Parse Error: syntax error while parsing: not_a_smiles
[11:49:35] SMILES Parse Error: check for mistakes around position 3:
[11:49:35] not_a_smiles
[11:49:35] ~~^
[11:49:35] SMILES Parse Error: Failed parsing SMILES 'not_a_smiles' for input: 'not_a_smiles'


# 3. Exercise 2 — First LLM Call

Now we introduce an LLM.

At this stage, the interaction is simply:

    Python → LLM → response

The LLM has no access to our scientific functions.

Therefore, it cannot execute:

- `calculate_area()`
- `calculate_molecular_weight()`

We will use the open-weight GPT-OSS 20B model through Groq.

Model:

    openai/gpt-oss-20b

This model supports reasoning and tool/function calling, which we will
use in the next exercises.

In [11]:
from groq import Groq

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

MODEL = "openai/gpt-oss-20b"

print("Groq client initialized.")
print("Model:", MODEL)

Groq client initialized.
Model: openai/gpt-oss-20b


## 3.1 Make the first LLM call

We will ask the LLM a simple scientific question.

Notice that we are not giving it any tools yet.

In [12]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": (
                "What is the difference between molecular weight "
                "and molecular mass?"
            )
        }
    ]
)

print(response.choices[0].message.content)

**Short answer**

In chemistry the terms *molecular mass* and *molecular weight* are used almost interchangeably, but strictly speaking:

| Term | What it really means | How it’s expressed | When the difference matters |
|------|----------------------|-------------------|----------------------------|
| **Molecular mass** | The mass of one molecule, measured in *unified atomic mass units* (u or amu). It’s a number with a unit. | 18.01528 u for water. | When you need the exact physical mass (e.g., for mass‑spectrometry). |
| **Molecular weight** | Historically the same number but written **without a unit**, as a dimensionless ratio to the mass of a hydrogen atom (1 u). It is *often* expressed as a gram‑per‑mole number (g mol⁻¹) when we talk about “grams per mole” – i.e., a *molar mass*. | 18.01528 (unitless) or 18.01528 g mol⁻¹ (the molar mass of water). | In everyday chemistry and biochemistry the term is fine; in physics or precision chemistry you should use “mass” with a unit.

---

#

## 3.2 What just happened?

The interaction was:

    Python
       ↓
    Groq API
       ↓
    GPT-OSS 20B
       ↓
    Text response
       ↓
    Python

There is still no agent.

The LLM simply generated a response to our question.

In [15]:
print(response)

ChatCompletion(id='chatcmpl-63fe87f6-e819-45d6-a40e-5029f9d07b47', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**Short answer**\n\nIn chemistry the terms *molecular mass* and *molecular weight* are used almost interchangeably, but strictly speaking:\n\n| Term | What it really means | How it’s expressed | When the difference matters |\n|------|----------------------|-------------------|----------------------------|\n| **Molecular mass** | The mass of one molecule, measured in *unified atomic mass units* (u or amu). It’s a number with a unit. | 18.01528\u202fu for water. | When you need the exact physical mass (e.g., for mass‑spectrometry). |\n| **Molecular weight** | Historically the same number but written **without a unit**, as a dimensionless ratio to the mass of a hydrogen atom (1\u202fu). It is *often* expressed as a gram‑per‑mole number (g\u202fmol⁻¹) when we talk about “grams per mole” – i.e., a *molar mass*. | 18.01528 (un

# 4. Exercise 3 — Give the LLM Tools

Our Python functions already work.

Now we want the LLM to know that these functions exist.

We do this using tool calling.

The LLM receives structured descriptions of our tools:

- tool name
- description
- parameters
- parameter types

The LLM can then decide:

    "I should use calculate_area."

But the LLM does not execute the Python function.

Our Python program executes it.

This is the transition from a normal LLM application to an
agent-like system.

## 4.1 Define the tool schemas

A tool schema tells the LLM:

- what the tool is called
- what it does
- what arguments it requires
- what type those arguments should be

In [16]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_area",
            "description": (
                "Calculate the area of a rectangle from "
                "its length and width."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "length": {
                        "type": "number",
                        "description": "Length of the rectangle."
                    },
                    "width": {
                        "type": "number",
                        "description": "Width of the rectangle."
                    }
                },
                "required": ["length", "width"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_molecular_weight",
            "description": (
                "Calculate molecular weight from a SMILES "
                "string using RDKit."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "smiles": {
                        "type": "string",
                        "description": (
                            "Molecular structure represented "
                            "as a SMILES string."
                        )
                    }
                },
                "required": ["smiles"]
            }
        }
    }
]

## 4.2 Ask the LLM to use a tool

We now give the tool definitions to the LLM.

The user asks a question that requires one of our tools.

The model should identify the appropriate tool and provide the
required arguments.

In [17]:
user_question = (
    "What is the area of a rectangle with length 12 and width 8?"
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": user_question
        }
    ],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

print(message)

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We should use the calculate_area function.', tool_calls=[ChatCompletionMessageToolCall(id='fc_3537ca7a-3eec-45c3-963d-84be05e12819', function=Function(arguments='{"length":12,"width":8}', name='calculate_area'), type='function')])


## 4.3 Inspect the raw tool call

Instead of immediately asking for a final answer, we inspect what the
LLM actually requested.

We expect something conceptually like:

    Tool:
        calculate_area

    Arguments:
        length = 12
        width = 8

The important point is that the LLM has not calculated the area.

It has requested that our Python program perform the calculation.

In [18]:
print("Tool calls:")
print(message.tool_calls)

Tool calls:
[ChatCompletionMessageToolCall(id='fc_3537ca7a-3eec-45c3-963d-84be05e12819', function=Function(arguments='{"length":12,"width":8}', name='calculate_area'), type='function')]


In [19]:
tool_call = message.tool_calls[0]

print("Tool name:")
print(tool_call.function.name)

print("\nArguments:")
print(tool_call.function.arguments)

print("\nTool call ID:")
print(tool_call.id)

Tool name:
calculate_area

Arguments:
{"length":12,"width":8}

Tool call ID:
fc_3537ca7a-3eec-45c3-963d-84be05e12819


# 5. Exercise 4 — Build the Mini Agent

We now combine everything.

The agent will perform this loop:

    User
      ↓
    LLM
      ↓
    Does the LLM want a tool?
      ↓
    Yes
      ↓
    Identify tool
      ↓
    Execute Python function
      ↓
    Return tool result to LLM
      ↓
    LLM generates final response
      ↓
    User

This is our first actual tool-using agent.

## 5.1 Create a tool registry

The LLM knows the names of our tools.

Our Python program needs to know which actual Python function
corresponds to each tool name.

The tool registry creates that mapping.

In [20]:
tool_registry = {
    "calculate_area": calculate_area,
    "calculate_molecular_weight": calculate_molecular_weight
}

## 5.2 Execute a requested tool

The LLM gives us:

- a tool name
- arguments

We use those values to call the corresponding Python function.

In [21]:
import json

def execute_tool(tool_name, arguments):
    """Execute a registered tool using the supplied arguments."""
    
    if tool_name not in tool_registry:
        raise ValueError(f"Unknown tool: {tool_name}")
    
    function = tool_registry[tool_name]
    
    return function(**arguments)

## 5.3 Build the mini agent

The agent will:

1. Send the user's question to the LLM.
2. Check whether the LLM requested a tool.
3. Execute the requested Python function.
4. Send the tool result back to the LLM.
5. Return the final response.

We are implementing the agent ourselves instead of using LangChain,
LangGraph, or another agent framework.

This keeps the underlying mechanism visible.

In [22]:
def run_agent(user_question):
    """Run the scientific calculator agent."""
    
    # -------------------------------------------------
    # Step 1: Send the user's question to the LLM
    # -------------------------------------------------
    messages = [
        {
            "role": "user",
            "content": user_question
        }
    ]
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    message = response.choices[0].message
    
    # -------------------------------------------------
    # Step 2: Check whether the LLM requested a tool
    # -------------------------------------------------
    if not message.tool_calls:
        return message.content
    
    # Add the assistant's tool-call message to the conversation
    messages.append(
        message.model_dump(exclude_none=True)
    )
    
    # -------------------------------------------------
    # Step 3: Execute the requested tool(s)
    # -------------------------------------------------
    for tool_call in message.tool_calls:
        
        tool_name = tool_call.function.name
        
        arguments = json.loads(
            tool_call.function.arguments
        )
        
        try:
            result = execute_tool(
                tool_name,
                arguments
            )
            
            tool_output = {
                "result": result
            }
            
        except Exception as e:
            tool_output = {
                "error": str(e)
            }
        
        # Add the tool result to the conversation
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_output)
            }
        )
    
    # -------------------------------------------------
    # Step 4: Send the tool result back to the LLM
    # -------------------------------------------------
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    # -------------------------------------------------
    # Step 5: Return the final answer
    # -------------------------------------------------
    return final_response.choices[0].message.content

# 6. Test the Agent

We will test three situations:

1. A question requiring the area tool.
2. A question requiring the molecular-weight tool.
3. An invalid molecular input.

These tests demonstrate that:

- the LLM selects the tool
- Python executes the tool
- RDKit performs the molecular calculation
- the result is returned to the LLM
- the LLM produces the final response

## 6.1 Test the area tool

In [23]:
question = (
    "What is the area of a rectangle with length 15 and width 4?"
)

answer = run_agent(question)

print(answer)

The area of the rectangle is **60** square units.


## 6.2 Test the molecular-weight tool

In [24]:
question = (
    "Calculate the molecular weight of ethanol. "
    "Its SMILES is CCO."
)

answer = run_agent(question)

print(answer)

The calculated molecular weight of ethanol (SMILES: CCO) is **46.069 g/mol**.


## 6.3 Test invalid input

Now we deliberately give the molecular-weight tool an invalid SMILES.

This is an important test because real scientific tools can produce
errors.

Our agent should not simply crash.

Instead, the error should be returned to the LLM so that it can explain
what happened to the user.

In [25]:
question = (
    "Calculate the molecular weight of this molecule: "
    "INVALID_SMILES"
)

answer = run_agent(question)

print(answer)

I’m sorry, but the string you provided (“INVALID_SMILES”) isn’t a valid SMILES representation of a molecule, so I can’t calculate a molecular weight. Please double‑check the SMILES and give me a correct one, and I’ll be happy to help!


# 7. What We Learned

This notebook introduced the fundamental components of an AI agent.

## 1. Python function

A normal scientific function performs a deterministic operation.

Example:

    calculate_molecular_weight("CCO")

## 2. LLM

The LLM interprets the user's request and decides what action may
be required.

## 3. Tool

A tool exposes a capability to the LLM.

Examples:

    calculate_area
    calculate_molecular_weight

## 4. Tool calling

The LLM produces a structured request containing:

- tool name
- arguments

Our Python program then executes the requested function.

## 5. Agent

The agent connects everything:

    User
      ↓
    LLM
      ↓
    Tool selection
      ↓
    Python / RDKit execution
      ↓
    Tool result
      ↓
    LLM
      ↓
    Final answer

## Central principle

The LLM is the coordinator.

The scientific software performs the scientific computation.

For scientific AI systems, this separation is extremely important.

# 7. What We Learned

This notebook introduced the fundamental components of an AI agent.

## 1. Python function

A normal scientific function performs a deterministic operation.

Example:

    calculate_molecular_weight("CCO")

## 2. LLM

The LLM interprets the user's request and decides what action may
be required.

## 3. Tool

A tool exposes a capability to the LLM.

Examples:

    calculate_area
    calculate_molecular_weight

## 4. Tool calling

The LLM produces a structured request containing:

- tool name
- arguments

Our Python program then executes the requested function.

## 5. Agent

The agent connects everything:

    User
      ↓
    LLM
      ↓
    Tool selection
      ↓
    Python / RDKit execution
      ↓
    Tool result
      ↓
    LLM
      ↓
    Final answer

## Central principle

The LLM is the coordinator.

The scientific software performs the scientific computation.

For scientific AI systems, this separation is extremely important.